In [ ]:
import mediapy as media
import imageio
from hydra.utils import instantiate
from einops import rearrange
from omegaconf import OmegaConf
import torch
import numpy as np
import random
# Register custom resolver to handle multiplication in OmegaConf interpolation
OmegaConf.register_new_resolver("mul", lambda x, y: float(x) * float(y))
OmegaConf.register_new_resolver("div", lambda a, b: float(a) / float(b))

In [ ]:
import torchvision.transforms.v2 as transforms
resize_to = (256, 256)
mean=(0.485, 0.456, 0.406)
std=(0.229, 0.224, 0.225)
rgb_transform = transforms.Compose(
            [
                transforms.Lambda(lambda x: x / 255.0),
                transforms.Lambda(lambda x: x.permute(0, 3, 1, 2)),  # (T, C, H, W)
                (
                    transforms.Resize(tuple(int(x) for x in resize_to))
                    if resize_to is not None
                    else transforms.Lambda(lambda x: x)
                ),  # resize
                (
                    transforms.Normalize(mean=mean, std=std, inplace=True)
                    if mean is not None
                    else transforms.Lambda(lambda x: x)
                ),  # normalize
            ]
        )

In [ ]:
snapshot_dir = "/svl/u/ravenh/lacwm/robot_world_models-raven-lam/projects/latent_action_models/data/experiments_0908/libero_sim_scratch_action_9/2025-11-24/19-41-14/"
config_path = f"{snapshot_dir}/.hydra"
print(config_path)

cfg = OmegaConf.load(config_path + "/config.yaml")

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("/svl/u/ravenh/lacwm/robot_world_models-raven-lam/projects/latent_action_models"))
os.environ['COSMOS_HOME'] = '/svl/u/ravenh/lacwm/Cosmos'


model = instantiate(cfg.model)
model = model.cuda().eval()
snapshot = torch.load(f"{snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
model.load_state_dict(snapshot["model"])

# Patch the tokenizer decode to handle dtype conversion for Cosmos
# if hasattr(model, 'rgb_tokenizer') and hasattr(model.rgb_tokenizer, 'decode'):
#     original_decode = model.rgb_tokenizer.decode
#     def patched_decode(z, out_shape):
#         # Convert to bfloat16 as required by Cosmos decoder
#         z = z.to(torch.bfloat16)
#         return original_decode(z, out_shape)
#     model.rgb_tokenizer.decode = patched_decode

val_dataloader = instantiate(cfg.val_data_loader)

In [ ]:
libero_dataloader = val_dataloader[0]

In [ ]:
for i, libero_batch in enumerate(libero_dataloader):
    if i > 1:
        break
    print(libero_batch.keys())
    break


In [ ]:
libero_batch['rgb'].shape

In [ ]:
input_rgb = libero_batch['rgb'].clone()
input_rgb[:,:] = input_rgb[:,0:1]

In [ ]:
model.ae_only = True  # Set to True to use autoencoder only
with torch.no_grad():
    with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
        predictions = model.visualize(libero_batch['rgb'].cuda(), libero_batch['actions'].cuda(), morphology_index=libero_batch['morphology_index'].cuda(), ee_action_dim=libero_batch['ee_action_dim'].cuda())

In [ ]:
model.ae_only = True  # Set to True to use autoencoder only
with torch.no_grad():
    with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
        predictions = model._generate_future(input_rgb.cuda(), libero_batch['actions'].cuda(), morphology_index=libero_batch['morphology_index'].cuda(), ee_action_dim=libero_batch['ee_action_dim'].cuda())

In [ ]:
#combine the predictions and the input_rgb
predictions = torch.cat([libero_batch['rgb'], predictions.cpu()], dim=-1)


In [ ]:
for i, prediction in enumerate(predictions):
    media.write_video(f"predictions_{i}.mp4", rearrange(prediction.to(torch.float32).cpu().numpy(), "t c h w -> t h w c") , fps=10)

In [ ]:
wm_action_chunk_size = 5
wm_history_size = 9
morphology_index = 5
ee_action_dim = 19
mutli_view = False
use_wrist_img = False

In [ ]:
import h5py
data = h5py.File("/viscam/projects/dexs2r/libero_init/libero_object_unseen/seed_1023/pick_up_the_red_coffee_mug_and_place_it_in_the_basket.hdf5", "r")

In [ ]:
data['data/'].keys()

In [ ]:
gt_actions = data['data/demo_35/actions'][:]

In [ ]:
img = data['data/demo_35/obs/agentview_rgb'][:][:,::-1,::-1]
wrist_img = data['data/demo_35/obs/eye_in_hand_rgb'][:][:,::-1, ::-1]
robot0_eef_pos = data['data/demo_35/obs/ee_pos']
robot0_eef_axis = data['data/demo_35/obs/ee_ori']
robot0_gripper_qpos = data['data/demo_35/obs/gripper_states']
task_description = 'open the top drawer and put the bowl inside'


In [ ]:
img[0]

In [ ]:
#save the first image
imageio.imwrite("first_image.png", (img[0]).astype(np.uint8))

In [ ]:
t = 10

In [ ]:
element = {
                            "observation/image": img[t],
                            "observation/wrist_image": wrist_img[t],
                            "observation/state": np.concatenate(
                                (
                                    robot0_eef_pos[t],
                                    robot0_eef_axis[t]  ,
                                    robot0_gripper_qpos[t],
                                )
                            ),
                            "prompt": str(task_description),
                        }

In [ ]:
from openpi_client import websocket_client_policy as _websocket_client_policy
client = _websocket_client_policy.WebsocketClientPolicy('10.79.12.193', 8000)

## Non batched Version

In [ ]:
##warm up
noise_scale = 10.0
noise = np.random.randn(50,32) * noise_scale
payload = {**element, "_noise": noise}
action_chunk = client.infer(payload)["actions"]

In [ ]:
all_action_chunks = []
for j in range(50):
    np.random.seed(j)
    random.seed(j)
    noise = np.random.randn(50, 32) * noise_scale
    payload = {**element, "_noise": noise}
    pi0_actions = client.infer(payload)["actions"]
    all_action_chunks.append(pi0_actions)

In [ ]:
# start_idx = t
# end_idx = t + wm_history_size*wm_action_chunk_size
# idx = np.arange(start_idx, end_idx, wm_action_chunk_size)
# idx[idx < 0] = 0

## Batched Version

In [ ]:
batch_size = 10
noise_scale = 1
noise = np.random.randn(batch_size, 50, 32) * noise_scale
for k,v in element.items():
    if isinstance(v, str):
        continue
    else:
        element[k] = v[None].repeat(batch_size, axis=0)
# element["prompt"] = np.array([str(task_description) for _ in range(batch_size)])
# del element["prompt"]

In [ ]:
payload = {**element, "_noise": noise}
pi0_actions = client.infer(payload)["actions"]

In [ ]:
all_action_chunks = np.array(pi0_actions)

In [ ]:
img_obs = img[:][t:t+1].repeat(wm_history_size, axis=0)
img_obs = torch.from_numpy(img_obs.copy()).float()  # (T, H, W, C), float32
img_obs_history = rgb_transform(img_obs)
wrist_img_obs = wrist_img[:][t:t+1].repeat(wm_history_size, axis=0)
wrist_img_obs = torch.from_numpy(wrist_img_obs.copy()).float()  # (T, H, W, C), float32
wrist_img_obs_history = rgb_transform(wrist_img_obs)

In [ ]:
if mutli_view:
    all_img_obs = torch.cat([img_obs_history, wrist_img_obs_history], dim=-1)
elif use_wrist_img:
    all_img_obs = wrist_img_obs_history
else:
    all_img_obs = img_obs_history


## Non-batched version

In [ ]:
def form_actions(action_chunk, use_gt_actions, gt_actions=None, t=None):
    if use_gt_actions:
        gripper_action = gt_actions[t,-1]
    else:
        gripper_action = action_chunk[0,-1]
    
    #fille with zero actions
    zero_action = np.zeros(7)
    zero_action[-1] = gripper_action
    zero_camera_action = np.zeros(9)
    zero_action_wcam = np.concatenate([zero_action, zero_camera_action], axis=0)
    action_input = zero_action_wcam[None].repeat(wm_history_size*wm_action_chunk_size, axis=0)
    
    if use_gt_actions:
        action_input[:action_chunk.shape[0],:7] = gt_actions[t:t+action_chunk.shape[0]]
    else:
        action_input[:action_chunk.shape[0],:7] = action_chunk
    return action_input


In [ ]:

def prepare_obs(action_input, all_img_obs, morphology_index,ee_action_dim):
    action_history_reshaped = action_input.reshape(-1, wm_action_chunk_size, action_input.shape[-1])

    cur_obs = {
        "rgb": all_img_obs[None],
        "actions": action_history_reshaped[None],
    }
    kwargs = {
        "morphology_index": np.array([morphology_index]),
        "ee_action_dim": np.array([ee_action_dim]),
    }
    for k,v in cur_obs.items():
        if isinstance(v, np.ndarray):
            cur_obs[k] = torch.from_numpy(v).to(torch.float32).cuda()
        else:
            cur_obs[k] = v.cuda()
    for k,v in kwargs.items():
        if isinstance(v, np.ndarray):
            kwargs[k] = torch.from_numpy(v).to(torch.float32).cuda()
        else:
            kwargs[k] = v.cuda()
    return cur_obs, kwargs

In [ ]:
use_gt_actions = False
end_imgs = []
for ac_idx, action_chunk in enumerate(all_action_chunks):
    
    action_input = form_actions(action_chunk[:wm_history_size*wm_action_chunk_size], use_gt_actions, gt_actions, t)
    cur_obs, kwargs = prepare_obs(action_input, all_img_obs, morphology_index,ee_action_dim)

    model.ae_only = True  # Set to True to use autoencoder only
    with torch.no_grad():
        predictions = model._generate_future(cur_obs["rgb"], cur_obs["actions"], **kwargs)
        

    C = predictions.shape[-3]
    device = predictions.device
    mean = torch.tensor(model.rgb_tokenizer._input_mean, device=device)
    std = torch.tensor(model.rgb_tokenizer._input_std, device=device)
    predictions = torch.clamp(
                    predictions * std.view(C, 1, 1) + mean.view(C, 1, 1), 0, 1
                )
    end_imgs.append(predictions[0,-1].to(torch.float32).cpu().numpy())

# media.write_video(f"predictions_pi.mp4", rearrange(predictions[0].to(torch.float32).cpu().numpy(), "t c h w -> t h w c") , fps=10)
media.write_video(f"predictions_pi.mp4", rearrange(np.stack(end_imgs), "t c h w -> t h w c") , fps=10)

In [ ]:
#save each end_img to a png file
for i, img in enumerate(end_imgs):
    imageio.imwrite(f"end_img_{i}.png", (img.transpose(1,2,0)*255).astype(np.uint8))

## Batched Version

In [ ]:
def get_batch_obs(all_action_chunks, all_img_obs, morphology_index,ee_action_dim):
    all_actions = np.array(all_action_chunks)[:,:wm_history_size*wm_action_chunk_size]
    batch_size = all_actions.shape[0]
    zero_camera_action = np.zeros((batch_size, wm_history_size*wm_action_chunk_size, 9))
    all_actions = np.concatenate([all_actions, zero_camera_action], axis=-1)
    action_history_reshaped = all_actions.reshape(batch_size, wm_history_size, wm_action_chunk_size, all_actions.shape[-1])
    cur_obs = {
        "rgb": all_img_obs[None].repeat(batch_size, 1, 1, 1, 1),  
        "actions": action_history_reshaped,
    }
    kwargs = {
        "morphology_index": np.repeat(np.array([morphology_index]), batch_size, axis=0),
        "ee_action_dim": np.repeat(np.array([ee_action_dim]), batch_size, axis=0),
    }
    for k,v in cur_obs.items():
        if isinstance(v, np.ndarray):
            cur_obs[k] = torch.from_numpy(v).to(torch.float32).cuda()
        else:
            cur_obs[k] = v.cuda()
    for k,v in kwargs.items():
        if isinstance(v, np.ndarray):
            kwargs[k] = torch.from_numpy(v).to(torch.float32).cuda()
        else:
            kwargs[k] = v.cuda()
    return cur_obs, kwargs



In [ ]:
cur_obs, kwargs = get_batch_obs(all_action_chunks, all_img_obs, morphology_index,ee_action_dim)

In [ ]:
gt_complete_actions = gt_actions[t:t+wm_history_size*wm_action_chunk_size].reshape(wm_history_size, wm_action_chunk_size, -1)
gt_complete_actions = np.concatenate([gt_complete_actions, np.zeros((wm_history_size, wm_action_chunk_size, 9))], axis=-1)

In [ ]:
cur_obs["actions"][0] = torch.from_numpy(gt_complete_actions).to(torch.float32).cuda()

In [ ]:
model.ae_only = True  # Set to True to use autoencoder only
all_predictions = []
batch_size = 4
with torch.no_grad():
    with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
        for i in range(0, cur_obs["actions"].shape[0], batch_size):
            batched_kwargs = {k:v[i:i+batch_size] for k,v in kwargs.items()}
            predictions = model._generate_future(cur_obs["rgb"][i:i+batch_size], cur_obs["actions"][i:i+batch_size], **batched_kwargs)
            all_predictions.append(predictions)
all_predictions_raw = torch.cat(all_predictions, dim=0)

In [ ]:

C = all_predictions_raw.shape[-3]
device = all_predictions_raw.device
mean = torch.tensor(model.rgb_tokenizer._input_mean, device=device)
std = torch.tensor(model.rgb_tokenizer._input_std, device=device)
all_predictions = torch.clamp(
                all_predictions_raw * std.view(C, 1, 1) + mean.view(C, 1, 1), 0, 1
            )


In [ ]:
all_predictions.shape

In [ ]:
for i, prediction in enumerate(all_predictions):
    media.write_video(f"predictions_pi_{i}.mp4", rearrange(prediction.to(torch.float32).cpu().numpy(), "t c h w -> t h w c") , fps=10)

In [ ]:
media.write_video(f"predictions_pi.mp4", rearrange(all_predictions[:,-1].to(torch.float32).cpu().numpy(), "t c h w -> t h w c") , fps=10)

## Select actions

In [ ]:
gt_images = data['data/demo_35/obs/agentview_rgb'][:][:,::-1,::-1]

In [ ]:
goal_images = gt_images[t+wm_history_size*wm_action_chunk_size-1]

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(goal_images)
plt.show()


In [ ]:
predicted_goal_images =rearrange(all_predictions[:,-1].to(torch.float32).cpu().numpy(), "t c h w -> t h w c") * 255

In [ ]:
#get the image latents
predicted_goal_images_input = rgb_transform(torch.from_numpy(predicted_goal_images).float().cuda())[:,None]
latents = model._forward_tokenizer_encode(predicted_goal_images_input)
# reconstructed_goal_images = model._forward_tokenizer_decode(latents, predicted_goal_images_input.shape)
# latents = model._forward_tokenizer_encode(all_predictions_raw[:,-1:])
## plot gt action
goal_images_input = rgb_transform(torch.from_numpy(goal_images.copy()).float().cuda()[None])[None]
gt_latents = model._forward_tokenizer_encode(goal_images_input)
# reconstructed_goal_images = model._forward_tokenizer_decode(gt_latents, goal_images_input.shape)

In [ ]:
latents.shape

In [ ]:
gt_latents.shape

In [ ]:
print(gt_latents.shape)
print(latents.shape)

In [ ]:
cosine_distance = torch.nn.functional.cosine_similarity(latents, gt_latents, dim=-1)
cosine_distance = cosine_distance.mean(dim=(1,2,3))
cosine_distance
## plot gt action

In [ ]:
distance = torch.nn.functional.mse_loss(latents, gt_latents, reduction='none')
distance = distance.mean(dim=(1,2,3,4))
distance

In [ ]:
index = distance.argmin()

## plot gt action

In [ ]:
predicted_indx = np.arange(t, t+wm_history_size*wm_action_chunk_size, wm_action_chunk_size)
gt_video_agent = img[:][predicted_indx]
gt_video_wrist = wrist_img[:][predicted_indx]
gt_video = np.concatenate([gt_video_agent, gt_video_wrist], axis=2)
media.write_video(f"gt_video.mp4", gt_video , fps=10)
